# Lesson 3.5: Nightwatch v0 finds the failing service
*Build an AI Agent from Scratch with LangGraph · Chapter 3*

**Goal:** run the full Nightwatch v0 (17 tools, guards on) on one hard incident, read every step, and grade it.

**What you will be able to do**
- sort v0's 17 tools into 9 reads, 7 writes and `finish`, and say why writes need evidence
- trace a run by hand, including how many messages each model call sends
- compute `grade()` by hand: right service, wrong writes, pass and clean
- explain why one green run is one sample, not a verdict

**How this notebook runs.** The run was recorded once with Groq (`v0_run.py`, record mode) and is replayed here from
`nightwatch/runs/cache`. No cell calls the API.

In [1]:
# Setup: find the course repo's nightwatch/ folder, load .env, check the key.
import os, sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

for cand in ["../../nightwatch", "../../../nightwatch", "nightwatch"]:   # relative to this notebook
    if (Path(cand) / "agent_v0.py").exists():
        NW = Path(cand).resolve()
        break
else:
    raise SystemExit("Can't find nightwatch/. Open this notebook from the course repo (lessons/<lesson>/notebook/).")
sys.path.insert(0, str(NW))

load_dotenv(find_dotenv(usecwd=True))          # the .env at the repo root, if there is one
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("nightwatch/ found:", (NW / "agent_v0.py").exists(), "| Python", sys.version.split()[0])
if HAS_KEY:
    print("GROQ_API_KEY loaded.")
else:
    print("No GROQ_API_KEY. Get a free key as in lesson 0.2 (console.groq.com -> API Keys) and put")
    print("GROQ_API_KEY=... in the .env file at the repo root.")
print("This notebook has no cell that calls the API: the run is a replay from nightwatch/runs/cache.")

nightwatch/ found: True | Python 3.13.16
GROQ_API_KEY loaded.
This notebook has no cell that calls the API: the run is a replay from nightwatch/runs/cache.


## 1. Recall: reading a looping trace (lesson 3.4)

A run fixes the shop but never calls `finish`: rule (b), *healthy but no finish*. In the baseline:

In [2]:
import json
import agent_v0
from brightcart.incidents import generate, fresh
from brightcart.sim import World

incs = {x.id: x for x in generate()}
rows = [json.loads(line) for line in open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8")]
nofin = [r for r in rows if not r["final"]]
print(f"no finish: {len(nofin)} | of those already healthy: {sum(r['healthy'] for r in nofin)}")

no finish: 46 | of those already healthy: 31


## 2. Key words

| Word | Meaning |
|---|---|
| incident | one broken state of the made-up shop, plus a pager alert |
| read tool | only looks (logs, health, deploys); changes nothing |
| write tool | changes the shop: a restart, a rollback, a new secret |
| finish | the last tool: the agent's conclusion, as JSON |
| hidden label | the true answer, which the agent never sees |
| grade | compares a finished run with that label |

## 3. The idea, then a hand trace

What v0 can see, and what it can change:

In [3]:
for t in agent_v0.TOOLS:
    f = t["function"]
    kind = "WRITE" if f["name"] in agent_v0.WRITE_TOOLS else ("end" if f["name"] == "finish" else "read")
    print(f"  {kind:5} {f['name']}({', '.join(f['parameters']['properties'])})")
print("\nSYSTEM:", agent_v0.SYSTEM)

  read  service_status(service)
  read  read_logs(service, n)
  read  recent_deploys()
  read  list_flags()
  read  config_changes()
  read  queue_depth()
  read  query_db(sql)
  read  read_runbook(topic)
  read  check_health()
  WRITE restart_service(service)
  WRITE rollback_deploy(service)
  WRITE scale_service(service, replicas)
  WRITE set_flag(flag, value)
  WRITE set_config(service, key, value)
  WRITE rotate_secret(service)
  WRITE clear_cache(service)
  end   finish(root_cause_service, cause, summary)

SYSTEM: You are Nightwatch, the on-call agent for the online shop Brightcart.
A pager alert woke you. Find the root cause, fix it with the smallest safe change,
confirm with check_health, then call finish.
Rules: read before you write. Never change something you have no evidence for.
If everything is healthy and the alert cleared, it is a false alarm: change nothing.


In [4]:
INC = incs["expired_secret-1"]
print(f"{INC.id} | difficulty {INC.difficulty} | PAGER: {INC.alert}")
print("recent deploys:", INC.deploys)
print("hidden label (the agent never sees it):", INC.label)

expired_secret-1 | difficulty hard | PAGER: checkout error rate above 5%
recent deploys: [{'t': '03:01', 'service': 'inventory', 'version': 'v2.4.9', 'by': 'ci'}]
hidden label (the agent never sees it): {'cause_service': 'payments', 'cause': 'expired_secret', 'fix': ['rotate_secret', 'payments', None]}


It is hard because the pager names **checkout**, the deploy points at **inventory**, and the cause is neither.
Guess first: how many steps will it need? The trace, by hand:

| entry | tool | what came back | messages sent in this call |
|---|---|---|---|
| 0 | `check_health` | checkout and payments unhealthy | 2 (system + pager) |
| 1 | `read_logs(checkout)` | an ERROR pointing upstream at payments | 4 |
| 2 | `read_logs(payments)` | `401 Unauthorized: key expired` | 6 |
| 3 | `rotate_secret(payments)` | done: **the only write**, after evidence | 8 |
| 4 | `check_health({"": ""})` | healthy True; the guard appends the nudge | 10 |
| 5 | `finish(expired_secret, payments)` | ok | 13 (10 + 2 + the nudge) |

Each step adds two messages (the assistant's tool call and its result); the nudge adds one.
Over 6 calls: 5,744 prompt + 555 completion tokens, about 6,300, over 90% of it prompt.

## 4. Write it yourself

**Count the messages.** Wrap `agent_v0.chat` (as in lesson 3.3) to record how many messages each call sends.

In [5]:
sent = []
real_chat = agent_v0.chat

def counting_chat(body, mode="record", sample=0):
    sent.append(len(body["messages"]))
    return real_chat(body, mode, sample)

agent_v0.chat = counting_chat
r = agent_v0.run(INC, mode="replay", sample=0, guards=True, verbose=True)
agent_v0.chat = real_chat
print("\nmessages per call:", sent)
print("finish():", json.dumps(r["final"], indent=1))
print(f"stop {r['stop_reason']} | steps {r['steps']} | healthy {r['healthy']} | writes {r['writes']}")
print(f"tokens prompt {r['usage']['prompt']} + completion {r['usage']['completion']} "
      f"({r['usage']['prompt'] / sum(r['usage'].values()):.0%} prompt)")

 0 check_health({}) -> {'healthy': False, 'unhealthy_services': ['checkout', 'payments'], 'queues_over_limit': {}}
 1 read_logs({"n": 20, "service": "checkout"}) -> ['03:01 INFO checkout handled request id=59367 in 76ms', '03:01 ERROR checkout upstream payments ret
 2 read_logs({"n": 20, "service": "payments"}) -> ['03:00 ERROR payments provider returned 401 Unauthorized: key expired', '03:00 ERROR payments provi
 3 rotate_secret({"service": "payments"}) -> rotate_secret(payments) done.
 4 check_health({"": ""}) -> {'healthy': True, 'unhealthy_services': [], 'queues_over_limit': {}}
 5 finish({"cause": "expired_secret", "root_cause_service": "payments", "summary": "Paymen) -> ok

messages per call: [2, 4, 6, 8, 10, 13]
finish(): {
 "cause": "expired_secret",
 "root_cause_service": "payments",
 "summary": "Payments provider secret expired causing 401 errors, leading to checkout error rate spike. Rotated secret resolved."
}
stop finish | steps 6 | healthy True | writes [('rotate_secret',

**grade() by hand.** Three things the model cannot argue with: health, the service, and the writes. Never the summary.
- right service: finish says payments, the label says payments → True
- wrong writes: 1 write − 1 fix that made the shop healthy = 0
- pass allows at most one wrong write, clean allows none → pass True, clean True

Now as code:

In [6]:
def my_grade(run):
    lab, fin = run["label"], run["final"] or {}
    right_service = fin.get("root_cause_service") == lab["cause_service"]
    if lab["fix"] is None:                                   # false alarm: say so, and change nothing
        ok = right_service and fin.get("cause") == "false_alarm" and not run["writes"]
        return {"pass": ok, "clean": ok, "right_service": right_service}
    wrong = len(run["writes"]) - (1 if run["healthy"] else 0)
    return {"pass": run["healthy"] and right_service and wrong <= 1,
            "clean": run["healthy"] and right_service and wrong == 0,
            "right_service": right_service}


print(my_grade(r))

{'pass': True, 'clean': True, 'right_service': True}


## 5. Library version + check

`agent_v0.grade` is the course's grader. Ours must agree on this run and on all 180 saved baseline runs
(each row gets its incident's label attached).

In [7]:
print("this run:", agent_v0.grade(r), "| same:", agent_v0.grade(r) == my_grade(r))
allr = [row | {"label": incs[row["incident"]].label} for row in rows]
agree = sum(my_grade(x) == agent_v0.grade(x) for x in allr)
print(f"agree on {agree} of {len(allr)} baseline runs | passes: {sum(my_grade(x)['pass'] for x in allr)}")

this run: {'pass': True, 'clean': True, 'right_service': True} | same: True
agree on 180 of 180 baseline runs | passes: 124


## 6. Where it shows up in agents

Real alerts fire where users feel the pain, not where the fault is. Checkout was the **symptom**; its logs pointed
**upstream** to payments; payments' logs named the **cause**. The inventory deploy was a decoy:

In [8]:
touched = {w[1] for w in r["writes"]}
reads = [t["args"].get("service") for t in r["trace"] if t["tool"] == "read_logs"]
print("logs read, in order:", reads, "| services written:", touched, "| inventory touched:", "inventory" in touched)

logs read, in order: ['checkout', 'payments'] | services written: {'payments'} | inventory touched: False


And one green run is one sample. The baseline ran v0 (no guards) on 60 incidents, 3 samples each:

In [9]:
import statistics as st
from collections import defaultdict
per = defaultdict(list)
for x in rows:
    per[x["incident"]].append(x["pass"])
print(f"pass rate {st.mean(x['pass'] for x in rows):.0%} | pass@3 {st.mean(any(v) for v in per.values()):.0%} | "
      f"pass^3 {st.mean(all(v) for v in per.values()):.0%} | mean tokens {st.mean(sum(x['usage'].values()) for x in rows):,.0f}")

pass rate 69% | pass@3 92% | pass^3 43% | mean tokens 11,060


69% (95% interval 61–77%, from `baseline.py`'s bootstrap). At least one of three passed on 92% of incidents, all three on
only 43%. About 11,000 tokens per baseline run against today's 6,300 is not like-for-like: the baseline had no guards.

## 7. Traps

**Trap 1: fixing the service the pager names.** Restart checkout on a fresh copy of the shop:

In [10]:
w = World(fresh(INC))
print(w.act("restart_service", "checkout"), "->", w.health())

restart_service(checkout) done. -> {'healthy': False, 'unhealthy_services': ['checkout', 'payments'], 'queues_over_limit': {}}


**Trap 2: trusting the finish summary.** The grader never reads it, so a wrong summary changes nothing:

In [11]:
fake = {**r, "final": {**r["final"], "summary": "It was a cosmic ray."}}
print(agent_v0.grade(fake))

{'pass': True, 'clean': True, 'right_service': True}


**Trap 3: calling v0 good after one green run.** The same incident in the baseline (no guards), three samples:

In [12]:
for x in rows:
    if x["incident"] == "expired_secret-1":
        print(f"s{x['sample']}: pass {x['pass']} | finish {(x['final'] or {}).get('root_cause_service')} | "
              f"healthy {x['healthy']} | writes {len(x['writes'])} | steps {x['steps']}")

s0: pass True | finish payments | healthy True | writes 1 | steps 6
s1: pass False | finish None | healthy True | writes 2 | steps 14
s2: pass True | finish payments | healthy True | writes 1 | steps 10


**Trap 4: counting only the right fix.** Every extra write costs the clean grade:

In [13]:
extra = {**r, "writes": [("restart_service", "checkout", None)] + r["writes"]}
print("restart checkout + rotate:", agent_v0.grade(extra))

restart checkout + rotate: {'pass': True, 'clean': False, 'right_service': True}


## 8. Exercises (no API: every check replays from the cache)

In [14]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

**Exercise 1 (the video's exercise).** Run v0 on another incident: `memory_leak-0`, sample 2, guards on,
`token_budget=60000`, replay mode. Store it in `r2`, print its trace, fill a trace table on paper, and **before**
printing `agent_v0.grade(r2)`, write your prediction into `predicted` (a dict with `pass`, `clean`, `right_service`).

In [15]:
r2, predicted = None, {}
# YOUR CODE HERE

In [16]:
def test1():
    assert r2 is not None and r2["incident"] == "memory_leak-0", "store the memory_leak-0 run in r2"
    assert r2["sample"] == 2 and r2["steps"] == 13, "sample 2, guards on: 13 steps"
    truth = agent_v0.grade(r2)
    for k in ("pass", "clean", "right_service"):
        assert predicted.get(k) == truth[k], f"your {k} prediction is off: count the writes and compare the service"
check(1, test1)

✗ Exercise 1 failed: store the memory_leak-0 run in r2


**Exercise 2.** Write `message_counts(n_calls, nudge_after)`: the number of messages each model call sends when every
step makes one tool call, and the nudge is appended after the step with index `nudge_after` (None for no nudge).

In [17]:
def message_counts(n_calls, nudge_after=None):
    # YOUR CODE HERE
    pass

In [18]:
def test2():
    assert message_counts(6, nudge_after=4) == [2, 4, 6, 8, 10, 13], "today's run: the nudge after entry 4"
    assert message_counts(6, nudge_after=4) == sent, "it should match what we counted on the wire"
    assert message_counts(3) == [2, 4, 6]
check(2, test2)

✗ Exercise 2 failed: today's run: the nudge after entry 4


**Exercise 3.** Write `wrong_writes(run)`: the number of writes that were not the fix, by grade's rule
(all writes minus one if the shop ended healthy).

In [19]:
def wrong_writes(run):
    # YOUR CODE HERE
    pass

In [20]:
def test3():
    assert wrong_writes(r) == 0, "today: one write, and it was the fix"
    assert wrong_writes(extra) == 1, "restart checkout + rotate: one wrong write"
    assert wrong_writes({"writes": [("restart_service", "web", None)], "healthy": False}) == 1, "unhealthy: no write counts as the fix"
check(3, test3)

✗ Exercise 3 failed: today: one write, and it was the fix


**Exercise 4.** Write `pass_at_k(rows)` returning `(pass@k, pass^k)` over incidents: the share of incidents where at least
one sample passed, and where all samples passed.

In [21]:
def pass_at_k(rows):
    # YOUR CODE HERE
    pass

In [22]:
def test4():
    at, hat = pass_at_k(rows)
    assert round(at, 2) == 0.92 and round(hat, 2) == 0.43, f"expected 92% and 43%, got {at:.0%} and {hat:.0%}"
    one = [{"incident": "a", "pass": True}, {"incident": "a", "pass": False}]
    assert pass_at_k(one) == (1.0, 0.0)
check(4, test4)

✗ Exercise 4 failed: TypeError: cannot unpack non-iterable NoneType object


## Solutions (try first!)

In [23]:
r2 = agent_v0.run(incs["memory_leak-0"], mode="replay", sample=2, guards=True, token_budget=60000)
for t in r2["trace"]:
    print(t["step"], t.get("tool") or t.get("error"), json.dumps(t.get("args"))[:50], "->", str(t.get("result"))[:40])
# three writes (scale, set_config, restart), healthy at the end, finish names inventory = the label
# wrong writes = 3 - 1 = 2 > 1  -> pass False, clean False, right_service True
predicted = {"pass": False, "clean": False, "right_service": True}
print(agent_v0.grade(r2))

0 service_status {"service": "inventory"} -> {'inventory': {'version': 'v3.0.0', 'rep
1 read_logs {"n": 20, "service": "inventory"} -> ['03:16 INFO inventory handled request i
2 recent_deploys {} -> []
3 config_changes {} -> []
4 scale_service {"replicas": 4, "service": "inventory"} -> scale_service(inventory) done.
5 check_health {} -> {'healthy': False, 'unhealthy_services':
6 list_flags {} -> {}
7 query_db {"sql": "SELECT * FROM deploys ORDER BY deployed_a -> ERROR: no such column: deployed_at
8 query_db {"sql": "SELECT * FROM deploys;"} -> []
9 set_config {"key": "memory_limit", "service": "inventory", "v -> set_config(inventory) done.
10 restart_service {"service": "inventory"} -> restart_service(inventory) done.
11 check_health {} -> {'healthy': True, 'unhealthy_services': 
12 finish {"cause": "bad_config", "root_cause_service": "inv -> ok
{'pass': False, 'clean': False, 'right_service': True}


In [24]:
def message_counts(n_calls, nudge_after=None):
    out, n = [], 2
    for i in range(n_calls):
        out.append(n)
        n += 2
        if i == nudge_after:
            n += 1
    return out

In [25]:
def wrong_writes(run):
    return len(run["writes"]) - (1 if run["healthy"] else 0)

In [26]:
def pass_at_k(rows):
    per = defaultdict(list)
    for x in rows:
        per[x["incident"]].append(x["pass"])
    return (sum(any(v) for v in per.values()) / len(per), sum(all(v) for v in per.values()) / len(per))